# 02_Silver_Medallion

Esta notebook lee el CSV raw de Bronze, realiza limpieza y transformación de datos utilizando una lógica de try-catch, y guarda el dataframe resultante como Parquet en la capa Silver

In [ ]:
!pip install pandas requests pyarrow openpyxl

In [ ]:
from pathlib import Path
import pandas as pd

### 1. Configuración de rutas y URLs

In [ ]:
BRONZE_PATH = Path('data/bronze/superstore_raw.csv')
SILVER_DIR = Path('data/silver')
SILVER_DIR.mkdir(parents=True, exist_ok=True)
SILVER_PATH = SILVER_DIR / 'superstore_clean.parquet'

### 2. Transformaciones

In [ ]:
# Leer desde el CSV de bronze
try:
    df = pd.read_csv(BRONZE_PATH, encoding='utf-8')
    print('Loaded bronze CSV with shape:', df.shape)
except Exception as exc:
    print('Error loading CSV:')
    print(str(exc))

In [ ]:
# Estandarizar nombres de columnas
try:
    # Standardize column names
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]
except Exception as exc:
    print('Error standardizing columns:')
    print(str(exc))

In [ ]:
# Limpiar y normalizar valores
try:
    # Clean and normalize values
    str_cols = df.select_dtypes(['object']).columns
    for col in str_cols:
        df[col] = df[col].astype('string').str.strip()
except Exception as exc:
    print('Error cleaning strings:')
    print(str(exc))

In [ ]:
# Parsear fechas y agregar campos derivados
try:
    # Parse dates and add derived fields
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')
    df['order_year'] = df['order_date'].dt.year
    df['order_month'] = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
except Exception as exc:
    print('Error parsing dates:')
    print(str(exc))

In [ ]:
# Validar datos y eliminar filas inválidas
try:
    # Remove rows without valid dates or sales
    before = len(df)
    df = df[df['order_date'].notna()]
    df = df[df['sales'].notna()]
    df = df[df['quantity'] > 0]
    df = df[df['sales'] >= 0]
    df = df.drop_duplicates()
    after = len(df)
    print(f'Removed {before-after} invalid or duplicate rows.')
except Exception as exc:
    print('Error removing invalid rows:')
    print(str(exc))

In [ ]:
# Convertir tipos numéricos y calcular margen de beneficio
try:
    # Numeric conversions
    df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce', downcast='integer')
    df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
    df['profit'] = pd.to_numeric(df['profit'], errors='coerce')
    df['profit_margin'] = df['profit'] / df['sales']
    df['profit_margin'] = df['profit_margin'].fillna(0.0)
except Exception as exc:
    print('Error in numeric conversions:')
    print(str(exc))

### 3. Almacenamiento

In [ ]:
# Guardar el DataFrame limpio en formato Parquet
try:
    df.to_parquet(SILVER_PATH, index=False)
    print('Saved cleaned silver parquet to:', SILVER_PATH.resolve())
    print('Silver dataset shape:', df.shape)
except Exception as exc:
    print('Error saving parquet:')
    print(str(exc))